# Average Multiple Days Together

## Imports and Versions

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from typing import Literal
from astropy import units as u

from pygsdata import GSData, plots, GSFlag
from pygsdata.register import gsregister

from edges import modeling as mdl
from edges.averaging import average_over_times, NsamplesStrategy
from edges.analysis.datamodel import add_model
from edges.filters import rfi_model_filter
from edges.alanmode import read_spec_txt
from edges.filters import filters
from edges.filters.filters import gsdata_filter
from edges.averaging import averaging

from edges_pipeline_utils import utils


In [2]:
plt.style.use("default")

In [3]:
utils.print_versions()


Versions: 
            read_acq: 1.2.0
            pygsdata: 0.2.3
      edges-analysis: 7.0.1.dev87+g7a9d94f69.d20250909


## Parameters and Data Loading

In [4]:
gathered_days_file: str = "gathered-days.gsh5"
alandir: str = "/home/smurray/data4/edges/alans-pipeline/scripts/H2CaseFieldData/"

raise_unmatching: bool = True
compare_alan: bool = True

In [5]:
# Parameters
raise_unmatching = False
compare_alan = False
gathered_days_file = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/work/d9/ddbfcc04e720afc62b8f9b9adc7ff8/gathered-days.gsh5"


In [6]:
gathered_days_file = Path(gathered_days_file)

In [7]:
data = GSData.from_file(gathered_days_file)

## Analysis and Averaging

### Model the spectra so we can average residuals only

When we perform the average over nights, we average the models and residuals separately (and the residuals receive frequency-dependent weights while the models do not). First, we model each night here.

In [8]:
data = add_model(data, model=mdl.models.PhysicalIono(spectral_index=-2.55, f_center=75.0, n_terms=5), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)

In [9]:
alanmodel = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_model_file.txt")

In [10]:
alan_dates = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_dates.txt")

In [11]:
alan_data = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_data.txt")

In [12]:
for i in range(data.ntimes):
    plt.plot(data.freqs, data.model[0,0,i] - alanmodel[i])
plt.xlabel("Frequency [MHz]")
plt.ylabel("Model Difference [K]")
plt.title("Difference in night-to-night models between edges-analysis and C-code")

Text(0.5, 1.0, 'Difference in night-to-night models between edges-analysis and C-code')

### Some Filters

Our first filter is an RMS filter (just thresholding a whole night based on its RMS to the fitted model)

In [13]:
alanfilt = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_flagfile.txt")

In [14]:
alflags = GSFlag(~(alanfilt[:, 1].astype(bool)), axes=("time",))

In [15]:
rms_data = filters.rms_filter(data, threshold=0.17, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM) 
rms_data_alan =  data.add_flags(flags=alflags, filt='applying Alans RMS filter flags')

In [16]:
if raise_unmatching:
    assert np.sum(~rms_data.flags['rms_filter'].flags[0,0] ^ alanfilt[:, 1].astype(bool))==0
elif compare_alan:
    print("Unmatched flags: ", np.sum(~rms_data.flags['rms_filter'].flags[0,0] ^ alanfilt[:, 1].astype(bool)))

In [17]:
rms_data = filters.prune_flagged_integrations(rms_data)
rms_data_alan = filters.prune_flagged_integrations(rms_data_alan)

In [18]:
print(rms_data.ntimes, rms_data_alan.ntimes)

67 62


Based on the model already fit, we perform simple RFI-flagging, where we flag any channel whose residual is larger than a threshold multiplied by the RMS of the residuals that night (over frequency).

In [19]:
filt_data = filters.rms_rfi_filter(rms_data, threshold=1.9, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)
filt_data_alan = filters.rms_rfi_filter(rms_data_alan, threshold=1.9)


Now, make sure that we flagged exactly the same channels/nights as the C-code

In [20]:
alanweights= np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_weights.txt")
weights= filt_data.flagged_nsamples[0,0] > 0

if raise_unmatching:
    assert np.sum(weights ^ alanweights.astype(bool))==0
elif compare_alan:
    print("Number of flags differing: ", np.sum(weights ^ alanweights.astype(bool)))

In [21]:
plots.plot_waterfall(filt_data);

### Average all nights

In [22]:
avg_data = average_over_times(
    filt_data, use_resids=True, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
)
avg_data_alan = average_over_times(
    filt_data_alan, use_resids=True, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
)

/data7/smurray/edges/edges-analysis/src/edges/averaging/lstbin.py:101: RuntimeWarning: invalid value encountered in divide
  mean_resids = sum_resids / ntot


### Last RFI Filter

In [23]:
avg_data = add_model(avg_data, model=mdl.Polynomial(offset=-2.5, n_terms=7, transform=mdl.ScaleTransform(scale=75.0)), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)
avg_data_alan = add_model(avg_data_alan, model=mdl.Polynomial(offset=-2.5, n_terms=7, transform=mdl.ScaleTransform(scale=75.0)), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES)

In [24]:
final_data = filters.rms_rfi_filter(avg_data, threshold=1.9)
final_data_alan = filters.rms_rfi_filter(avg_data_alan, threshold=1.9)

In [25]:
final_data = add_model(final_data, model=mdl.LinLog(n_terms=5))
final_data_alan = add_model(final_data_alan, model=mdl.LinLog(n_terms=5))

## Inspect Final Results

In [26]:
alan_fl = f"{alandir}/final_average_latest.txt"
alan = np.genfromtxt(alan_fl, usecols=(1, 3, 6, 9, 12), names=('freq', 'tant', 'model', 'resid', 'weight'))

In [27]:
linlog = mdl.LinLog(n_terms=5).at(x=final_data.freqs)

In [28]:
ourfit = linlog.fit(ydata=final_data.data[0,0,0], weights=final_data.flagged_nsamples[0,0,0]>0)
ourfit_alan = linlog.fit(ydata=final_data_alan.data[0,0,0], weights=final_data_alan.flagged_nsamples[0,0,0]>0)
alfit = linlog.fit(ydata=alan['tant'], weights=alan['weight'])

In [29]:
def plot_final_data(final_data, label: str):
    fig, ax = plt.subplots(3, 1, sharex=True, figsize=(10, 8), constrained_layout=True)

    ax[0].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, final_data.data[0,0,0], np.nan), label=label)
    ax[0].plot(final_data.freqs, np.where(alan['weight']>0, alan['tant'], np.nan), label='C-code', ls='--')
    ax[0].legend()
    ax[0].text(0.95, 0.9, "Spectrum", transform=ax[0].transAxes, ha='right', fontweight='bold')

    for i in range(3):
        ax[i].set_ylabel("Temperature [K]")
        
    ttmin = data.times.min().datetime.timetuple()
    ttmax = data.times.max().datetime.timetuple()

    fig.suptitle(f"Final Averaged Spectrum: {rms_data.ntimes} days [{ttmin.tm_year}:{ttmin.tm_yday:>03} -- {ttmax.tm_year}:{ttmax.tm_yday:>03}]")

    ax[1].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, ourfit.residual, np.nan))

    ax[1].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, alfit.residual, np.nan), ls='--')
    ax[1].text(0.95, 0.9, "FG Residuals (LinLog, 5-term)", transform=ax[1].transAxes, ha='right', fontweight='bold')


    ax[2].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, final_data.data[0,0,0] - alan['tant'], np.nan), color='b')
    ax[2].set_xlabel("Frequency [MHz]")
    ax[2].text(0.95, 0.9, "Absolute Difference between pipelines", transform=ax[2].transAxes, ha='right', fontweight='bold');


In [30]:
plot_final_data(final_data, 'edges-analysis')

In [31]:
plot_final_data(final_data_alan, 'edges-analysis w/ final nights from B18')

We ensure that we have all the same flags as the C-code:

In [32]:
if raise_unmatching:
    assert np.sum((final_data.flagged_nsamples > 0) ^ alan['weight'].astype(bool))==0
elif compare_alan:
    print("Unmatched flags: ", np.sum((final_data.flagged_nsamples > 0) ^ alan['weight'].astype(bool)))

In [33]:
plt.plot(final_data.freqs, final_data.flagged_nsamples[0,0,0])
plt.xlabel("Frequency [MHz]")
plt.ylabel("Nsamples");

## Write out the data

In [34]:
final_data.write_gsh5(gathered_days_file.parent / "averaged_spectrum.gsh5");